In [ ]:
"""
Sequential Purchase Recommender  — v2 (Markov orders 1–4)
==========================================================
Models the sequence of product purchases to recommend the next product.

Approach stack:
  1. 1st-order Markov Chain    — P(next | last)
  2. 2nd-order Markov Chain    — P(next | last-2, last-1)
  3. 3rd-order Markov Chain    — P(next | last-3, last-2, last-1)
  4. 4th-order Markov Chain    — P(next | last-4, ..., last-1)
  5. Association Rules         — confidence + lift
  6. Stage-aware popularity    — what % of accounts buy X at stage k
  7. Hybrid scorer             — weighted combination of all signals
  8. Leave-last-out evaluation — Hit@K, MRR@K, NDCG@K
  9. Ablation study            — each signal tested individually

Run:  python sequential_recommender_v2.py
Output: sequential_recommender_v2_results.xlsx
"""

import pandas as pd
import numpy as np
from collections import defaultdict, Counter
from itertools import combinations
import warnings
warnings.filterwarnings("ignore")

# ── 1. CONFIG ────────────────────────────────────────────────────────────────
DATA_PATH    = "C:/Users/14082/Dropbox/SkiGeni Practicum/2026/Ignite PiedPiper Santa Clara/Source Data/Purchase_Data.csv"
OUTPUT_PATH  = "C:/Users/14082/Dropbox/SkiGeni Practicum/2026/sequential_recommender_results_V3.xlsx"
TOPK        = [1, 3, 5, 10]
MIN_SEQ_LEN = 2
MIN_SUPPORT = 5
LEVEL       = "Product_Name"

# Weights for all 7 signals — auto-normalised so they sum to 1.
# Higher-order Markov chains receive lower default weights because they
# have sparser coverage (fewer accounts have long enough histories to
# populate order-3 and order-4 states).
WEIGHTS = {
    "markov1":   0.35,
    "markov2":   0.20,
    "markov3":   0.10,
    "markov4":   0.05,
    "ar_conf":   0.15,
    "ar_lift":   0.08,
    "stage_pop": 0.07,


    "markov1":   0.15,
    "markov2":   0.15,
    "markov3":   0.15,
    "markov4":   0.15,
    "ar_conf":   0.15,
    "ar_lift":   0.15,
    "stage_pop": 0.1,
}

TOP_N_RECS = 10


# ── 2. DATA LOADING & PREPROCESSING ─────────────────────────────────────────
def load_data(path: str, level: str) -> pd.DataFrame:
    df = pd.read_csv(path, dtype={"Account_ID": str})
    df["Won_Date"] = pd.to_datetime(df["Won_Date"])
    df = df.sort_values(["Account_ID", "Won_Date"])
    df = df.drop_duplicates(subset=["Account_ID", "Won_Date", level])
    return df


def build_sequences(df: pd.DataFrame, level: str) -> pd.DataFrame:
    seqs = (
        df.sort_values(["Account_ID", "Won_Date"])
        .groupby("Account_ID")[level]
        .apply(list)
        .reset_index()
    )
    seqs.columns = ["Account_ID", "sequence"]
    seqs = seqs[seqs["sequence"].apply(len) >= MIN_SEQ_LEN].copy()
    seqs["seq_len"] = seqs["sequence"].apply(len)
    return seqs


def train_test_split_last(seqs: pd.DataFrame):
    """Leave-last-out: train on all but last purchase, test on last."""
    train = seqs.copy()
    train["train_seq"]  = seqs["sequence"].apply(lambda s: s[:-1])
    train["test_label"] = seqs["sequence"].apply(lambda s: s[-1])
    return train[train["train_seq"].apply(len) >= 1]


# ── 3. MARKOV CHAIN (any order) ──────────────────────────────────────────────
class MarkovChain:
    """
    Nth-order Markov chain for sequential product recommendation.

    self.order controls how many prior purchases form the 'state':
      order=1 → state is the single last purchase         (1 item)
      order=2 → state is the last 2 purchases as a tuple  (2 items)
      order=3 → state is the last 3 purchases as a tuple  (3 items)
      order=4 → state is the last 4 purchases as a tuple  (4 items)

    The fit() loop always steps through each sequence extracting every
    consecutive window of length (order + 1):
        window = [seq[i], seq[i+1], ..., seq[i+order-1], seq[i+order]]
        state  = window[:order]   (the conditioning context)
        next   = window[order]    (what we want to predict)

    Because higher orders require longer histories, the number of
    distinct states grows with order but coverage (fraction of accounts
    that have a matching state at prediction time) falls:

        order=1 → 608  states,  ~100% coverage
        order=2 → 5,794 states, ~71%  coverage
        order=3 → 11,550 states, ~55% coverage
        order=4 → 13,794 states, ~38% coverage
    """

    def __init__(self, order: int = 1):
        # Memory size: how many previous purchases to look at (Default is 1)
        self.order = order
        
        # Prepares a blank dictionary to tally up historical product transitions.
        # defaultdict(Counter) is a trick that automatically creates a new tally
        # bucket on the fly if a product has never been seen before (prevents crash)
        self.transitions: dict = defaultdict(Counter)


    def fit(self, sequences):
        for seq in sequences:
            # range(len(seq) - self.order) ensures seq[i + self.order]
            # is always a valid index — we stop self.order positions
            # before the end of the sequence.
            for i in range(len(seq) - self.order):

                # Build the state (conditioning context):
                #   order=1 → a single string  e.g. "Product_238"
                #   order=2 → a 2-tuple        e.g. ("Product_238", "Product_441")
                #   order=3 → a 3-tuple        e.g. ("Product_238", "Product_441", "Product_103")
                #   order=4 → a 4-tuple        e.g. (...)
                if self.order == 1:
                    state = seq[i]
                else:
                    state = tuple(seq[i : i + self.order])

                next_item = seq[i + self.order]
                self.transitions[state][next_item] += 1

        # Convert raw counts → probabilities (maximum likelihood estimate)
        # P̂(next | state) = count(state → next) / Σ_k count(state → k)
        self.probs: dict = {}
        for state, counts in self.transitions.items():
            total = sum(counts.values())
            self.probs[state] = {k: v / total for k, v in counts.items()}

        return self

    def predict(self, history: list) -> dict:
        """
        Return {product: probability} using the last `self.order` items
        in history as the conditioning state.

        Returns {} (empty dict) when:
          - history is shorter than self.order (not enough context)
          - the state was never seen in training (unseen n-gram)
        Both cases are handled by the HybridRecommender fallback below.
        """
        if len(history) < self.order:
            return {}

        if self.order == 1:
            state = history[-1]
        else:
            # Slice the last `self.order` items and make a tuple
            state = tuple(history[-self.order:])

        return self.probs.get(state, {})


# ── 4. ASSOCIATION RULES ─────────────────────────────────────────────────────
class AssociationRules:
    def __init__(self, min_support: int = MIN_SUPPORT):
        self.min_support = min_support

    def fit(self, sequences):
        item_counts: Counter = Counter()
        pair_counts: Counter = Counter()
        n_accounts = len(sequences)

        for seq in sequences:
            unique = set(seq)
            for item in unique:
                item_counts[item] += 1
            for a, b in combinations(sorted(unique), 2):
                pair_counts[(a, b)] += 1

        self.rules: dict = defaultdict(list)
        self.n_accounts = n_accounts
        self.item_counts = item_counts

        for (a, b), count in pair_counts.items():
            if count < self.min_support:
                continue
            for ant, con in [(a, b), (b, a)]:
                conf = count / item_counts[ant]
                lift = conf / (item_counts[con] / n_accounts)
                self.rules[ant].append(
                    {"product": con, "confidence": conf, "lift": lift, "support": count}
                )

        for ant in self.rules:
            self.rules[ant].sort(key=lambda x: -x["lift"])

        all_lifts = [r["lift"] for rlist in self.rules.values() for r in rlist]
        self.max_lift = max(all_lifts) if all_lifts else 1.0
        return self

    def predict_confidence(self, history: list) -> dict:
        last = history[-1]
        return {r["product"]: r["confidence"] for r in self.rules.get(last, [])}

    def predict_lift(self, history: list) -> dict:
        last = history[-1]
        return {r["product"]: r["lift"] / self.max_lift
                for r in self.rules.get(last, [])}


# ── 5. STAGE-AWARE POPULARITY ────────────────────────────────────────────────
class StagePopularity:
    def fit(self, sequences):
        stage_counts: dict = defaultdict(Counter)
        for seq in sequences:
            for stage, item in enumerate(seq, start=1):
                stage_counts[stage][item] += 1

# It looks through every customer's history. The enumerate(..., start=1)
# function acts like a counter. It looks at the first item and tags it stage=1. It
# looks at the second item and tags it stage=2. Then, it adds a tally mark to its
# bucket. "Ah, Product A was bought at Stage 1. Let's add a tally to the Stage 1
# bucket for Product A."

        self.stage_probs: dict = {}
        for stage, counts in stage_counts.items():
            total = sum(counts.values())
            self.stage_probs[stage] = {k: v / total for k, v in counts.items()}
        self.max_stage = max(self.stage_probs.keys()) if self.stage_probs else 0
        return self

    def predict(self, history: list) -> dict:
        next_stage = len(history) + 1
        stage = min(next_stage, self.max_stage)
        return self.stage_probs.get(stage, {})


# ── 6. HYBRID SCORER ─────────────────────────────────────────────────────────
class HybridRecommender:
    """
    Combines all signals into a single ranked recommendation.

    Fallback cascade for higher-order Markov chains
    ─────────────────────────────────────────────────
    When a higher-order state is not found in the training data, its
    weight is redistributed to the next lower order rather than wasted.
    The cascade works inward:

        order=4 missing → its weight shifts to order=3
        order=3 missing → its weight (+ any absorbed from 4) shifts to order=2
        order=2 missing → its weight (+ any absorbed from 3,4) shifts to order=1

    So order=1 always ends up with at least its own weight, and often
    more when higher-order contexts are unavailable.  This means the
    model gracefully degrades for accounts with short histories.

    Example for an account with only 1 purchase in history:
        - order=4 → no prediction (history too short) → weight cascades down
        - order=3 → no prediction (history too short) → weight cascades down
        - order=2 → no prediction (history too short) → weight cascades down
        - order=1 → has prediction → receives all 4 Markov weights
    """

    def __init__(self, weights: dict = WEIGHTS):
        self.weights   = self._normalise(weights)
        self.mc1       = MarkovChain(order=1)
        self.mc2       = MarkovChain(order=2)
        self.mc3       = MarkovChain(order=3)
        self.mc4       = MarkovChain(order=4)
        self.ar        = AssociationRules()
        self.stage_pop = StagePopularity()

    @staticmethod
    def _normalise(w: dict) -> dict:
        total = sum(w.values())
        return {k: v / total for k, v in w.items()}

    def fit(self, sequences):
        self.mc1.fit(sequences)
        self.mc2.fit(sequences)
        self.mc3.fit(sequences)
        self.mc4.fit(sequences)
        self.ar.fit(sequences)
        self.stage_pop.fit(sequences)
        return self

    def score(self, history: list) -> dict:
        """
        Compute hybrid scores for all candidate products.

        Cascading fallback logic:
          Each higher-order model is tried; if it returns an empty dict
          (state unseen or history too short), its weight is passed down
          to the next lower order.
        """
        w = self.weights

        m1  = self.mc1.predict(history)
        m2  = self.mc2.predict(history)
        m3  = self.mc3.predict(history)
        m4  = self.mc4.predict(history)
        arc = self.ar.predict_confidence(history)
        arl = self.ar.predict_lift(history)
        sp  = self.stage_pop.predict(history)

        # ── Cascading weight redistribution ──────────────────────────────
        # Start from the highest order and cascade unused weight downward.
        # 'pool' accumulates weight from orders that had no prediction.
        pool = 0.0

        # order=4: if no prediction, pool its weight
        if not m4:
            pool += w["markov4"]
            eff_w4 = 0.0
        else:
            eff_w4 = w["markov4"]

        # order=3: absorbs any pooled weight from order=4
        if not m3:
            pool += w["markov3"]
            eff_w3 = 0.0
        else:
            eff_w3 = w["markov3"] + pool
            pool = 0.0          # pool has been absorbed

        # order=2: absorbs any remaining pooled weight
        if not m2:
            pool += w["markov2"]
            eff_w2 = 0.0
        else:
            eff_w2 = w["markov2"] + pool
            pool = 0.0

        # order=1: always has a prediction (unless completely new product);
        # absorbs all remaining pooled weight
        eff_w1 = w["markov1"] + pool
        # ─────────────────────────────────────────────────────────────────

        candidates = set(m1) | set(m2) | set(m3) | set(m4) | \
                     set(arc) | set(arl) | set(sp)

        scores = {}
        for p in candidates:
            scores[p] = (
                eff_w1             * m1.get(p,  0.0)
                + eff_w2           * m2.get(p,  0.0)
                + eff_w3           * m3.get(p,  0.0)
                + eff_w4           * m4.get(p,  0.0)
                + w["ar_conf"]     * arc.get(p, 0.0)
                + w["ar_lift"]     * arl.get(p, 0.0)
                + w["stage_pop"]   * sp.get(p,  0.0)
            )
        return scores

    def recommend(self, history: list, top_n: int = TOP_N_RECS) -> list:
        scores = self.score(history)
        return sorted(scores.items(), key=lambda x: -x[1])[:top_n]


# ── 7. EVALUATION ─────────────────────────────────────────────────────────────
def hit_at_k(ranked, truth, k):
    return 1.0 if truth in [p for p, _ in ranked[:k]] else 0.0

def rr_at_k(ranked, truth, k):
    for i, (p, _) in enumerate(ranked[:k]):
        if p == truth:
            return 1.0 / (i + 1)
    return 0.0

def ndcg_at_k(ranked, truth, k):
    for i, (p, _) in enumerate(ranked[:k]):
        if p == truth:
            return 1.0 / np.log2(i + 2)
    return 0.0

def evaluate(model, test_df, ks=TOPK):
    metrics = {k: {"hit": [], "mrr": [], "ndcg": []} for k in ks}
    for _, row in test_df.iterrows():
        hist  = row["train_seq"]
        truth = row["test_label"]
        recs  = model.recommend(hist, top_n=max(ks))
        for k in ks:
            metrics[k]["hit"].append(hit_at_k(recs, truth, k))
            metrics[k]["mrr"].append(rr_at_k(recs, truth, k))
            metrics[k]["ndcg"].append(ndcg_at_k(recs, truth, k))

    rows = []
    for k in ks:
        rows.append({
            "K":           k,
            "Hit@K":       np.mean(metrics[k]["hit"]),
            "MRR@K":       np.mean(metrics[k]["mrr"]),
            "NDCG@K":      np.mean(metrics[k]["ndcg"]),
            "N_evaluated": len(metrics[k]["hit"]),
        })
    return pd.DataFrame(rows)


# ── 8. MAIN ───────────────────────────────────────────────────────────────────
def main():
    print("=" * 60)
    print("Sequential Purchase Recommender  v2  (Markov orders 1–4)")
    print("=" * 60)

    # ── Load & preprocess ────────────────────────────────────────────────────
    print(f"\n[1/6] Loading data ...")
    df = load_data(DATA_PATH, LEVEL)
    print(f"      {len(df):,} clean events | {df['Account_ID'].nunique():,} accounts")

    seqs_df = build_sequences(df, LEVEL)
    print(f"      {len(seqs_df):,} accounts with {MIN_SEQ_LEN}+ purchases")

    split      = train_test_split_last(seqs_df)
    train_seqs = split["train_seq"].tolist()
    print(f"      {len(split):,} accounts in evaluation set")

    # Coverage stats per order
    print("\n      Markov order coverage (accounts with enough history):")
    for order in [1, 2, 3, 4]:
        eligible = sum(1 for s in train_seqs if len(s) >= order)
        print(f"        order={order}: {eligible:,} / {len(train_seqs):,} "
              f"({eligible/len(train_seqs):.1%})")

    # ── Fit full model ────────────────────────────────────────────────────────
    print("\n[2/6] Fitting models ...")
    full_model = HybridRecommender(weights=WEIGHTS)
    full_model.fit(seqs_df["sequence"].tolist())
    print(f"      Markov-1 states: {len(full_model.mc1.probs):,}")
    print(f"      Markov-2 states: {len(full_model.mc2.probs):,}")
    print(f"      Markov-3 states: {len(full_model.mc3.probs):,}")
    print(f"      Markov-4 states: {len(full_model.mc4.probs):,}")
    total_rules = sum(len(v) for v in full_model.ar.rules.values())
    print(f"      AR rules:        {total_rules:,}")

    # ── Hybrid evaluation ─────────────────────────────────────────────────────
    print("\n[3/6] Evaluating hybrid model (leave-last-out) ...")
    eval_model = HybridRecommender(weights=WEIGHTS)
    eval_model.mc1       = full_model.mc1
    eval_model.mc2       = full_model.mc2
    eval_model.mc3       = full_model.mc3
    eval_model.mc4       = full_model.mc4
    eval_model.ar        = full_model.ar
    eval_model.stage_pop = full_model.stage_pop

    eval_df = evaluate(eval_model, split)
    print(eval_df.to_string(index=False))

    # ── Ablation study ────────────────────────────────────────────────────────
    print("\n[4/6] Running ablation study ...")
    ar_abl    = AssociationRules().fit(train_seqs)
    stage_abl = StagePopularity().fit(train_seqs)
    mc_models = {o: MarkovChain(order=o).fit(train_seqs) for o in [1, 2, 3, 4]}

    ablation_rows = []
    for k_eval in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
        models_to_test = [
            ("Popularity (baseline)", lambda h, s=stage_abl:
                sorted(s.predict(h).items(), key=lambda x: -x[1])),
            ("Markov-1 only",  lambda h, m=mc_models[1]:
                sorted(m.predict(h).items(), key=lambda x: -x[1])),
            ("Markov-2 only",  lambda h, m=mc_models[2]:
                sorted(m.predict(h).items(), key=lambda x: -x[1])),
            ("Markov-3 only",  lambda h, m=mc_models[3]:
                sorted(m.predict(h).items(), key=lambda x: -x[1])),
            ("Markov-4 only",  lambda h, m=mc_models[4]:
                sorted(m.predict(h).items(), key=lambda x: -x[1])),
            ("AR (confidence)", lambda h, a=ar_abl:
                sorted(a.predict_confidence(h).items(), key=lambda x: -x[1])),
            ("AR (lift)",       lambda h, a=ar_abl:
                sorted(a.predict_lift(h).items(), key=lambda x: -x[1])),
            ("Hybrid (all)",    lambda h:
                eval_model.recommend(h, top_n=20)),
        ]
        for name, predict_fn in models_to_test:
            hits, mrrs, ndcgs = [], [], []
            for _, row in split.iterrows():
                hist  = row["train_seq"]
                truth = row["test_label"]
                try:
                    recs = predict_fn(hist)[:k_eval]
                except Exception:
                    recs = []
                hits.append(hit_at_k(recs, truth, k_eval))
                mrrs.append(rr_at_k(recs, truth, k_eval))
                ndcgs.append(ndcg_at_k(recs, truth, k_eval))
            ablation_rows.append({
                "Model":   name,
                "K":       k_eval,
                "Hit@K":   round(np.mean(hits),  4),
                "MRR@K":   round(np.mean(mrrs),  4),
                "NDCG@K":  round(np.mean(ndcgs), 4),
            })

    ablation_df = pd.DataFrame(ablation_rows)
    print(ablation_df.to_string(index=False))


###########################################################
    # ── Per-account recommendations ───────────────────────────────────────────
    print("\n[5/6] Generating per-account recommendations (Testing Past Predictions) ...")
    recs_rows = []
    
    for _, row in seqs_df.iterrows():
        full_history = row["sequence"]
        
        # We can't do a hide-and-seek test on accounts with only 1 purchase
        if len(full_history) < 2:
            continue
            
        # The item we are trying to guess
        true_last_purchase = full_history[-1]
        
        # The history the model is allowed to see (everything EXCEPT the last purchase)
        test_history = full_history[:-1]
        
        acct_id = row["Account_ID"]
        
        # Generate recommendations using ONLY the test_history
        recs  = full_model.recommend(test_history, top_n=TOP_N_RECS)
        owned = set(test_history)

        for rank, (product, score) in enumerate(recs, start=1):
            mask   = df[LEVEL] == product
            group  = df.loc[mask, "Product_Group"].iloc[0]  if mask.any() else ""
            family = df.loc[mask, "Product_Family"].iloc[0] if mask.any() else ""

            # Individual signal contributions (Make sure to use test_history here too!)
            m1s  = full_model.mc1.predict(test_history).get(product, 0)
            m2s  = full_model.mc2.predict(test_history).get(product, 0)
            m3s  = full_model.mc3.predict(test_history).get(product, 0)
            m4s  = full_model.mc4.predict(test_history).get(product, 0)
            arcs = full_model.ar.predict_confidence(test_history).get(product, 0)
            arls = full_model.ar.predict_lift(test_history).get(product, 0)
            sps  = full_model.stage_pop.predict(test_history).get(product, 0)

            acct_name = df.loc[df["Account_ID"] == acct_id, "Account_Name"].iloc[0] \
                        if (df["Account_ID"] == acct_id).any() else ""

            recs_rows.append({
                "Account_ID":       acct_id,
                "Account_Name":     acct_name,
                "Sequence_Length":  len(test_history),
                "Last_Purchase":    true_last_purchase,  # This remains the true target
                "Rank":             rank,
                "Recommended":      product,             # This is the model's guess
                "Product_Group":    group,
                "Product_Family":   family,
                "Already_Owned":    product in owned,
                "Hybrid_Score":     round(score, 6),
                "Score_Markov1":    round(m1s,  6),
                "Score_Markov2":    round(m2s,  6),
                "Score_Markov3":    round(m3s,  6),
                "Score_Markov4":    round(m4s,  6),
                "Score_AR_Conf":    round(arcs, 6),
                "Score_AR_Lift":    round(arls, 6),
                "Score_Stage_Pop":  round(sps,  6),
            })

    recs_df = pd.DataFrame(recs_rows)
    print(f"      {len(recs_df):,} test recommendation rows for eligible accounts")

#######################################################














    # ── Coverage summary ──────────────────────────────────────────────────────
    # How often did each Markov order actually contribute?
    coverage_rows = []
    for order in [1, 2, 3, 4]:
        col = f"Score_Markov{order}"
        active = (recs_df[col] > 0).sum()
        coverage_rows.append({
            "Markov_Order":        order,
            "State_Count":         len(getattr(full_model, f"mc{order}").probs),
            "Rec_Rows_With_Signal": active,
            "Coverage_%":          round(active / len(recs_df) * 100, 1),
        })
    coverage_df = pd.DataFrame(coverage_rows)
    print("\n      Markov order contribution in final recommendations:")
    print(coverage_df.to_string(index=False))

    # ── Write Excel ────────────────────────────────────────────────────────────
    print("\n[6/6] Writing Excel output ...")
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter
    from openpyxl.formatting.rule import ColorScaleRule

    wb = Workbook()

    H_FILL      = PatternFill("solid", fgColor="1F4E79")
    H_FONT      = Font(name="Arial", bold=True, color="FFFFFF", size=10)
    SH_FILL     = PatternFill("solid", fgColor="2E75B6")
    SH_FONT     = Font(name="Arial", bold=True, color="FFFFFF", size=10)
    BODY        = Font(name="Arial", size=10)
    ALT_FILL    = PatternFill("solid", fgColor="EBF3FA")
    YLW_FILL    = PatternFill("solid", fgColor="FFF2CC")
    GRN_FILL    = PatternFill("solid", fgColor="E2EFDA")
    HLT_FILL    = PatternFill("solid", fgColor="FFF2CC")

    def write_df(ws, df_in, start_row=1):
        for ci, col in enumerate(df_in.columns, 1):
            c = ws.cell(row=start_row, column=ci, value=col)
            c.fill = H_FILL; c.font = H_FONT
            c.alignment = Alignment(horizontal="center", vertical="center")
        last_row = start_row
        for ri, row in enumerate(df_in.itertuples(index=False), start_row + 1):
            for ci, val in enumerate(row, 1):
                c = ws.cell(row=ri, column=ci, value=val)
                c.font = BODY
                c.alignment = Alignment(horizontal="left", vertical="center")
                if ri % 2 == 0: c.fill = ALT_FILL
            last_row = ri
        return last_row

    def autofit(ws, min_w=8, max_w=40):
        for col in ws.columns:
            ln = max((len(str(c.value)) if c.value else 0) for c in col)
            ws.column_dimensions[get_column_letter(col[0].column)].width = \
                min(max(ln + 2, min_w), max_w)

    # ── Sheet 1: Cover ────────────────────────────────────────────────────────
    ws0 = wb.active
    ws0.title = "Cover"
    ws0.sheet_view.showGridLines = False
    ws0["B2"] = "Sequential Purchase Recommender v2 — Markov Orders 1–4"
    ws0["B2"].font = Font(name="Arial", bold=True, size=15, color="1F4E79")
    ws0["B3"] = f"Granularity: {LEVEL}  |  Orders: 1, 2, 3, 4  |  Evaluation: leave-last-out"
    ws0["B3"].font = Font(name="Arial", size=10, color="595959")
    ws0["B5"] = "Tab"
    ws0["B5"].font = SH_FONT; ws0["B5"].fill = SH_FILL
    ws0["C5"] = "Contents"
    ws0["C5"].font = SH_FONT; ws0["C5"].fill = SH_FILL
    tabs = [
        ("Evaluation",      "Hit@K, MRR@K, NDCG@K for the full hybrid model"),
        ("Ablation",        "Each signal tested individually including Markov 3 & 4"),
        ("Coverage",        "How often each Markov order contributed a non-zero signal"),
        ("Recommendations", "Top-10 recommendations per account with score decomposition"),
        ("Methodology",     "Hybrid formula, cascading fallback logic, weight tuning guide"),
    ]
    for i, (tab, desc) in enumerate(tabs, 6):
        ws0.cell(row=i, column=2, value=tab).font = \
            Font(name="Arial", size=10, color="2E75B6", bold=True)
        ws0.cell(row=i, column=3, value=desc).font = Font(name="Arial", size=10)
    ws0.column_dimensions["B"].width = 22
    ws0.column_dimensions["C"].width = 70

    # ── Sheet 2: Evaluation ───────────────────────────────────────────────────
    ws1 = wb.create_sheet("Evaluation")
    ws1.sheet_view.showGridLines = False
    ws1["A1"] = "Hybrid Model (orders 1–4) — Leave-Last-Out Evaluation"
    ws1["A1"].font = Font(name="Arial", bold=True, size=13, color="1F4E79")
    ws1["A2"] = f"N = {len(split):,} accounts"
    ws1["A2"].font = Font(name="Arial", size=10, color="595959")
    ed = eval_df.copy()
    ed["Hit@K"]  = ed["Hit@K"].map("{:.2%}".format)
    ed["MRR@K"]  = ed["MRR@K"].map("{:.4f}".format)
    ed["NDCG@K"] = ed["NDCG@K"].map("{:.4f}".format)
    write_df(ws1, ed, start_row=4)
    autofit(ws1)

    # ── Sheet 3: Ablation ─────────────────────────────────────────────────────
    ws2 = wb.create_sheet("Ablation")
    ws2.sheet_view.showGridLines = False
    ws2["A1"] = "Per-Signal Ablation — Including Markov-3 and Markov-4"
    ws2["A1"].font = Font(name="Arial", bold=True, size=13, color="1F4E79")
    ws2["A2"] = "Each model tested individually. 'Hybrid' uses all 7 signals with cascading fallback."
    ws2["A2"].font = Font(name="Arial", size=10, color="595959")
    ad = ablation_df.copy()
    for col in ["Hit@K", "MRR@K", "NDCG@K"]:
        ad[col] = ad[col].map("{:.2%}".format)
    write_df(ws2, ad, start_row=4)
    autofit(ws2)

    # ── Sheet 4: Coverage ─────────────────────────────────────────────────────
    ws_cov = wb.create_sheet("Coverage")
    ws_cov.sheet_view.showGridLines = False
    ws_cov["A1"] = "Markov Order Coverage"
    ws_cov["A1"].font = Font(name="Arial", bold=True, size=13, color="1F4E79")
    ws_cov["A2"] = ("Coverage % = fraction of recommendation rows where the order had a "
                    "non-zero signal. Lower = sparser. Sparse orders still help via the "
                    "cascading fallback.")
    ws_cov["A2"].font = Font(name="Arial", size=10, color="595959")
    write_df(ws_cov, coverage_df, start_row=4)
    autofit(ws_cov)

    # ── Sheet 5: Recommendations ──────────────────────────────────────────────
    ws3 = wb.create_sheet("Recommendations")
    ws3.sheet_view.showGridLines = False
    ws3["A1"] = f"Per-Account Recommendations — Top {TOP_N_RECS}"
    ws3["A1"].font = Font(name="Arial", bold=True, size=13, color="1F4E79")
    ws3["A2"] = "Score_Markov3 and Score_Markov4 show 0 when the n-gram state was not seen in training."
    ws3["A2"].font = Font(name="Arial", size=10, color="595959")
    recs_display = recs_df.sort_values(["Account_ID", "Rank"])
    last_rec_row = write_df(ws3, recs_display, start_row=4)
    ao_col = list(recs_display.columns).index("Already_Owned") + 1
    for ri in range(5, last_rec_row + 1):
        if ws3.cell(row=ri, column=ao_col).value is True:
            for ci in range(1, len(recs_display.columns) + 1):
                ws3.cell(row=ri, column=ci).fill = HLT_FILL
    ws3.freeze_panes = "A5"
    autofit(ws3)

    # ── Sheet 6: Methodology ──────────────────────────────────────────────────
    ws_m = wb.create_sheet("Methodology")
    ws_m.sheet_view.showGridLines = False
    ws_m.column_dimensions["B"].width = 100
    lines = [
        ("B2",  "Methodology — v2 Hybrid Scorer with Markov Orders 1–4",
                Font(name="Arial", bold=True, size=14, color="1F4E79")),
        ("B4",  "Hybrid Score Formula",
                Font(name="Arial", bold=True, size=12, color="2E75B6")),
        ("B5",  "score(p) = w1·P(p|last) + w2·P(p|last-2,last-1) + w3·P(p|last-3,..,last-1) + w4·P(p|last-4,..,last-1)",
                Font(name="Courier New", size=10)),
        ("B6",  "         + w5·conf(last→p) + w6·lift_norm(last→p) + w7·P(p|stage_k)",
                Font(name="Courier New", size=10)),
        ("B8",  "Cascading Fallback Logic",
                Font(name="Arial", bold=True, size=12, color="2E75B6")),
        ("B9",  ("When a higher-order Markov state is not found (n-gram unseen or history too short), "
                 "its weight cascades to the next lower order rather than being wasted:"),
                Font(name="Arial", size=10)),
        ("B10", "  order=4 missing → weight shifts to order=3",
                Font(name="Arial", size=10, color="595959")),
        ("B11", "  order=3 missing → weight (incl. anything cascaded from 4) shifts to order=2",
                Font(name="Arial", size=10, color="595959")),
        ("B12", "  order=2 missing → weight (incl. anything cascaded from 3,4) shifts to order=1",
                Font(name="Arial", size=10, color="595959")),
        ("B13", "  order=1 always receives at least its own weight, often more.",
                Font(name="Arial", size=10, color="595959")),
        ("B15", "Default Weights (edit WEIGHTS dict at top of script to tune)",
                Font(name="Arial", bold=True, size=12, color="2E75B6")),
    ]
    for key, val in WEIGHTS.items():
        lines.append(("B" + str(16 + list(WEIGHTS.keys()).index(key)),
                      f"  {key}: {val:.2f}  (normalised: {val/sum(WEIGHTS.values()):.2%})",
                      Font(name="Arial", size=10)))
    lines += [
        ("B24", "Sparsity vs. Accuracy Trade-off",
                Font(name="Arial", bold=True, size=12, color="2E75B6")),
        ("B25", ("Markov-3 and Markov-4 have far more distinct states than order 1 or 2 "
                 "(11,550 and 13,794 vs 608 and 5,794) but lower coverage (55% and 38% of "
                 "accounts have a matching state). They contribute most for accounts with "
                 "long, predictable purchase patterns."),
                Font(name="Arial", size=10)),
        ("B27", "When to reduce order-3/4 weights:",
                Font(name="Arial", bold=True, size=10)),
        ("B28", "  → Most accounts have fewer than 4 purchases (little coverage gain)",
                Font(name="Arial", size=10)),
        ("B29", "  → You want faster inference (each order adds a dict lookup per prediction)",
                Font(name="Arial", size=10)),
        ("B30", "When to increase order-3/4 weights:",
                Font(name="Arial", size=10, bold=True)),
        ("B31", "  → Many accounts have long histories (5+ purchases) with consistent patterns",
                Font(name="Arial", size=10)),
        ("B32", "  → Evaluation shows Hit@K improves when you increase those weights",
                Font(name="Arial", size=10)),
    ]
    for ref, text, font in lines:
        c = ws_m[ref]; c.value = text; c.font = font
        c.alignment = Alignment(wrap_text=True, vertical="top")

    wb.save(OUTPUT_PATH)
    print(f"\n✓ Saved: {OUTPUT_PATH}")
    print("  Sheets: Cover | Evaluation | Ablation | Coverage | Recommendations | Methodology")


if __name__ == "__main__":
    main()



# Account_4533's sequence
seq = df[df['Account_ID']=='00100000002Mv09AAC'].sort_values('Won_Date')['Product_Group'].tolist()
print("Sequence:", seq)  # 12 purchases: Sc_i_th×5, As_p_tw_h_h×3, As_x_tw_h_h×4


Sequential Purchase Recommender  v2  (Markov orders 1–4)

[1/6] Loading data ...
      28,649 clean events | 3,981 accounts
      3,510 accounts with 2+ purchases
      3,510 accounts in evaluation set

      Markov order coverage (accounts with enough history):
        order=1: 3,510 / 3,510 (100.0%)
        order=2: 3,242 / 3,510 (92.4%)
        order=3: 2,662 / 3,510 (75.8%)
        order=4: 1,958 / 3,510 (55.8%)

[2/6] Fitting models ...
      Markov-1 states: 608
      Markov-2 states: 5,794
      Markov-3 states: 11,550
      Markov-4 states: 13,794
      AR rules:        8,626

[3/6] Evaluating hybrid model (leave-last-out) ...
 K    Hit@K    MRR@K   NDCG@K  N_evaluated
 1 0.682906 0.682906 0.682906         3510
 3 0.856980 0.758500 0.783745         3510
 5 0.937037 0.776918 0.816825         3510
10 0.982621 0.783235 0.831802         3510

[4/6] Running ablation study ...
                Model  K  Hit@K  MRR@K  NDCG@K
Popularity (baseline)  1 0.1028 0.1028  0.1028
        Markov

NameError: name 'df' is not defined

: 